# HiSparse 入门 (4/8)：缓存算子家族 —— gather / invalidate / pack / scatter

> **一句话结论**：resolver 出计划之后，由一组缓存算子执行搬运、失效、组装与写回；A5 上缓存类算子用 Triton，resolver 用原生 AscendC，注意力复用既有 SFA。

## 1. 四个基础动作

| 动作 | 职责 | 通俗解释 |
| --- | --- | --- |
| gather | 把缺失行从 Host/Resident 字节拷贝到目标池 | 把仓库/书架上的书取到桌面 |
| invalidate | 写槽后清掉所有指向旧行号的索引（置 `-1`） | 书换了内容，旧书签全部作废 |
| pack | 把 KV 分量组装成标准 656B 行 | 把散页装进标准信封 |
| scatter | 把组装好的行写回池 | 信封放回对应格子 |

## 2. A5 上的实现分工

- **Resolver**：原生 AscendC SIMT AIV 内核（见上一篇）；
- **缓存/失效/打包/散布**：Triton 算子（`ops/triton/hisparse.py`）；
- **注意力**：复用原生 SFA，不另写注意力算术：
  - BF16 KV：`npu_sparse_flash_attention`；
  - C8 量化 KV：`npu_kv_quant_sparse_flash_attention_vllm`。

## 3. 图捕获安全约束（很重要）

这些算子会被录进 CUDA/Ascend 图，因此必须满足：

- 图捕获内**不做堆分配**；scratch / workspace 提前预分配并复用；
- 输出张量预先分配好；
- 捕获路径上**不做 host 同步**；
- 相同输入产出稳定布局，保证图可以重放。

## 4. 小结

- 四个动作：gather 取行、invalidate 作废旧索引、pack 装行、scatter 写回；
- A5：Triton 跑缓存算子，AscendC 跑 resolver，注意力走既有 SFA；
- 所有算子受图捕获安全约束；
- 下一篇拆 656B 行的字节布局。

**下一篇**：`05-c8-row-format-a5.ipynb`
